# Importing the necessary packages/modules

In [73]:
from IPython.display import display
import pandas as pd


# Loading and previewing the data

In [74]:
#loading the final raw
df =pd.read_csv('../data/raw/cwlagos_listings_raw.csv')
df.head()

,type,kind,price,title,location,beds,baths,agent,contact,listing_url
0,For Rent,Apartment,"₦20,000,000",4 Units of 3 Bedroom Apartments in Lekki phase 1,Lekki Phase 1,3.0,3.0,Chinenye,+234 816 911 2079,https://cwlagos.com/property/4-units-of-3-bedr...
1,For sale,Detached Duplex,"₦1,500,000,000",Luxury 5 Bedroom Fully Detached House in Lekki...,Lekki Phase 1,5.0,5.0,Chinenye,+234 816 911 2079,https://cwlagos.com/property/luxury-5-bedroom-...
2,For Rent,Apartment,"₦45,000,000",Furnished 3 Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,+234 816 631 5298,https://cwlagos.com/property/furnished-3-bedro...
3,For Rent,Apartment,"₦89,408,528",Luxury 3-Bedroom Apartment in Ikoyi,Banana Island,3.0,3.0,Adaeze,+234 816 631 5298,https://cwlagos.com/property/luxury-3-bedroom-...
4,For Rent,Commercial,"₦30,000,000",4 Bedroom Apartment for Commercial Use in VI,Victoria Island,NaN,NaN,Nadi,+2349062511345,https://cwlagos.com/property/4-bedroom-apartme...


In [75]:
print(df.shape)

(652, 10)


In [76]:
df.columns

Index(['type', 'kind', 'price', 'title', 'location', 'beds', 'baths', 'agent',
       'contact', 'listing_url'],
      dtype='str')

# Handling the duplicate entries & inconsistent naming

## handling duplicate urls(unique identifier)

In [77]:
# previewing to see the total no of duplicates
df['listing_url'].duplicated().sum()

np.int64(0)

In [78]:
""" We see that there are no duplicates in the listing_url column, which is a good unique identifier for each listing. """

' We see that there are no duplicates in the listing_url column, which is a good unique identifier for each listing. '

## handling duplicate locations

In [79]:
df['location'].value_counts().to_frame()

,count
location,
Lekki Phase 1,118
Ikoyi,106
Victoria Island,86
Ikate,56
Lekki,43
Oniru,38
Ikota,37
Banana Island,30
Osapa,25


In [80]:
# these are unique locations in lagos, but most of them are specific areas within the larger districts, so we will need to do some cleaning to group them together
district_mapping = {
    # ikoyi parent
    "Ikoyi": "Ikoyi",
    "Old Ikoyi": "Ikoyi",
    "Banana Island": "Ikoyi",
    "Parkview": "Ikoyi",
    "Osborne Foreshore": "Ikoyi",

    # Victoria Island Parent 
    "Victoria Island": "Victoria Island",
    "Eko Atlantic": "Victoria Island",
    "Oniru": "Victoria Island",

    # Lekki Parent 
    "Lekki": "Lekki",
    "Lekki Phase 1": "Lekki",
    "Ikate": "Lekki",
    "Osapa": "Lekki",
    "Ologolo": "Lekki",
    "Chevron": "Lekki",
    "Ikota": "Lekki",
    "Orchid": "Lekki",
    "Orchid, Lekki": "Lekki",
    "Pinnock Beach Estate": "Lekki",

    # Ajah Parent 
    "Ajah": "Ajah"
}

In [81]:
# now creating a new column called 'district' and mapping the locations to their respective districts
df['district'] = df['location'].map(district_mapping)
df['district'].value_counts().to_frame()

,count
district,
Lekki,335
Ikoyi,168
Victoria Island,126
Ajah,17


## previewing the "kind" column

In [82]:
#looks very consistent
df['kind'].value_counts().to_frame()

,count
kind,
Apartment,288
Detached Duplex,126
Terrace,80
Semi Detached,39
Commercial,36
Mixed-Use Land,32
Maisonette,22
Penthouse,17
Residential Land,6


## previewing the "type" column

In [83]:
df['type'].value_counts().to_frame()

,count
type,
For Rent,321
For Sale,274
Land,47
For sale,9


In [84]:
# fixing iconsistency with the sale columns
df['type'] = df['type'].replace('For sale', 'For Sale')
df['type'].value_counts().to_frame()

,count
type,
For Rent,321
For Sale,283
Land,47


## previewing the price column

In [85]:
df['price'].head()

0       ₦20,000,000
1    ₦1,500,000,000
2       ₦45,000,000
3       ₦89,408,528
4       ₦30,000,000
Name: price, dtype: str

In [86]:
# we see the price column is in string format and has the Naira symbol and commas, I'll need to clean this to convert it to a numeric format for analysis
df['price'] = df['price'].str.replace('₦', '', regex=False)
df['price'] = df['price'].str.replace(',', '', regex=False)
df['price'] = df['price'].astype(float)
df['price'].head()

ValueError: could not convert string to float: '650000000 / 1500000000'

In [87]:
#taking a look at the specific outlier
df[df['price']  == '650000000 / 1500000000']

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
138,For Sale,Apartment,650000000 / 1500000000,4 Bedroom Flat & 5 Bedroom Penthouse Maisonett...,Victoria Island,4.0,4.0,Ifunanya,+234 706 399 0727,https://cwlagos.com/property/4-bedroom-flat-an...,Victoria Island


In [88]:
pd.set_option('display.max_colwidth', None)

In [89]:
# checking the url to see if there are any clues about the price
display(df[df['price']  == '650000000 / 1500000000']['listing_url'])

138    https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104
Name: listing_url, dtype: str

In [90]:
# i see that the listing is actually two listings in one, I'll need to create the two separate listings and drop the original one
extra_listing = [{
    "title": "5 Bedroom Penthouse Maisonette",
    "location": "Oniru",
    "type": "For Sale",
    "kind": "Maisonette",
    "beds": 5.0,
    "baths": 5.0,
    "price": 1500000000,
    "district": "Victoria Island",
    "agent": "Ifunanya",
    "contact": "+234 706 399 0727",
    "listing_url": "https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104"
}, 
{
    "title": "4 Bedroom Flat (235sqm) new development at Oniru, Victoria Island",
    "location": "Oniru",
    "district": "Victoria Island",
    "type": "For Sale",
    "kind": "Apartment",
    "beds": 4.0,
    "baths": 4.0,
    "price": 650000000,
    "agent": "Ifunanya",
    "contact": "+234 706 399 0727",
    "listing_url": "https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104",
}]

extra_listing_df = pd.DataFrame(extra_listing)
df = pd.concat([df, extra_listing_df], ignore_index=True)

In [91]:
"""dropping the original listing with the combined price"""
df = df[df['price'] != '650000000 / 1500000000']
df.shape

(653, 11)

In [92]:
#finally, converting the price column to float format for analysis
df['price'] = df['price'].astype(float)
df['price'].dtype

dtype('float64')

# Final review of the cleaned data

In [93]:
df.describe(include='all')

,type,kind,price,title,location,beds,baths,agent,contact,listing_url,district
count,652,653,6.530000e+02,653,649,564.000000,564.000000,653,653,653,647
unique,3,11,NaN,505,21,NaN,NaN,17,17,652,4
top,For Rent,Apartment,NaN,2 Bedroom Apartment in Lekki,Lekki Phase 1,NaN,NaN,Jennifer,+234 704 808 9361,https://cwlagos.com/property/4-bedroom-flat-and-5-bedroom-penthouse-maisonette-in-oniru-victoria-island-cw08104,Lekki
freq,321,288,NaN,11,118,NaN,NaN,89,89,2,335
mean,NaN,NaN,4.379179e+08,NaN,NaN,3.581560,3.581560,NaN,NaN,NaN,NaN
std,NaN,NaN,1.110910e+09,NaN,NaN,2.294966,2.294966,NaN,NaN,NaN,NaN
min,NaN,NaN,1.500000e+05,NaN,NaN,1.000000,1.000000,NaN,NaN,NaN,NaN
25%,NaN,NaN,2.500000e+07,NaN,NaN,3.000000,3.000000,NaN,NaN,NaN,NaN
50%,NaN,NaN,1.200000e+08,NaN,NaN,4.000000,4.000000,NaN,NaN,NaN,NaN
75%,NaN,NaN,4.500000e+08,NaN,NaN,4.000000,4.000000,NaN,NaN,NaN,NaN


In [94]:
df.to_csv('../data/cleaned/lagos_real_estate_market_data_cleaned.csv', index=False)